# 3. Models

Part 3 of 5. How well do the three input sets predict annual SPF and flag weak heat pumps, and which model works best?

Models:
- Ridge regression (`RidgeCV`)
- Random forest (`RandomForestRegressor`)
- LightGBM (`LGBMRegressor`)
- Automatic choice: in each split, picks one of the three using the training homes only

Testing: 5-fold cross-validation repeated 10 times, so every home is predicted by a model that never saw it.

Main metric: the 25% of homes with the lowest predicted SPF are flagged, for example for an inspection. What share of them is truly weak? Random flagging gives 22.7%.

Rule fixed before seeing the results: Ridge is the main model unless another model beats it on flagged precision or rank correlation with a 95% interval fully above zero.

Runtime is about 20 minutes, mostly the random forest.

## Step 1: Setup

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')

import os
DATA_DIR = '/content/drive/MyDrive/EoH_project'      # the raw EoH files
OUT_DIR = DATA_DIR + '/q1_paper'                     # everything this paper series makes
FIG_DIR = OUT_DIR + '/figures'
TABLE_DIR = OUT_DIR + '/tables'
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TABLE_DIR, exist_ok=True)

import json
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import rankdata
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, GridSearchCV, cross_val_score
from sklearn.metrics import r2_score
from lightgbm import LGBMRegressor
warnings.filterwarnings('ignore')

plt.rcParams.update({'font.size': 10, 'axes.titlesize': 11, 'axes.labelsize': 10, 'legend.fontsize': 8,
                     'axes.spines.top': False, 'axes.spines.right': False})
BLUE = '#2f6f9f'
ORANGE = '#c96a14'
GREY = '#8a8f98'
GREEN = '#3f8f5a'
MAKER = {'Daikin': 'Manufacturer A', 'Mitsubishi': 'Manufacturer B', 'Vaillant': 'Manufacturer C'}
MAKER_COLOR = {'Daikin': '#2f6f9f', 'Mitsubishi': '#8a5aa8', 'Vaillant': '#5b8c5a'}

B = 2000          # bootstrap resamples for the 95% intervals
REPEATS = 10      # repeated 5-fold splits
print('ready')

## Step 2: Read notebook 2 outputs

`q1_table.csv, q1_input_sets.json`

In [ ]:
data = pd.read_csv(OUT_DIR + '/q1_table.csv')
with open(OUT_DIR + '/q1_input_sets.json') as f:
    input_sets = json.load(f)

spf = data['SPF'].values
weak = spf < 2.5
base_rate = 100 * weak.mean()
print('homes:', len(data), '| weak:', weak.sum(), f'| random flagging: {base_rate:.1f}%')
print({k: len(v) for k, v in input_sets.items()})

## Step 3: Models and tuning

Random forest and LightGBM settings are tuned inside each training fold with an inner 5-fold split (nested cross-validation), so test homes are never used for tuning.

- Random forest (200 trees): `min_samples_leaf` 5, 10, 20 and `max_features` 0.33, 1.0
- LightGBM (300 small trees, learning rate 0.03): `num_leaves` 3, 7 and `min_child_samples` 10, 20, 40. Trees are kept small because each fold has only about 340 training homes.

Ridge chooses its penalty inside the training homes, after standardising the inputs.

In [ ]:
inner_cv = KFold(n_splits=5, shuffle=True, random_state=0)


def make_ridge():
    return make_pipeline(StandardScaler(), RidgeCV(alphas=[0.1, 1, 3, 10, 30, 100, 300]))


def make_forest():
    forest = RandomForestRegressor(n_estimators=200, random_state=0, n_jobs=1)
    settings = {'min_samples_leaf': [5, 10, 20], 'max_features': [0.33, 1.0]}
    return GridSearchCV(forest, settings, cv=inner_cv, scoring='neg_mean_squared_error', n_jobs=-1)


def make_lightgbm():
    booster = LGBMRegressor(n_estimators=300, learning_rate=0.03, subsample=0.8, subsample_freq=1,
                            colsample_bytree=0.8, random_state=0, n_jobs=1, verbose=-1)
    settings = {'num_leaves': [3, 7], 'min_child_samples': [10, 20, 40]}
    return GridSearchCV(booster, settings, cv=inner_cv, scoring='neg_mean_squared_error', n_jobs=-1)


model_names = ['Ridge', 'Random forest', 'LightGBM']
makers = {'Ridge': make_ridge, 'Random forest': make_forest, 'LightGBM': make_lightgbm}
print('3 models ready')

## Step 4: Helper functions

- Flagged precision: share of truly weak homes among the 25% with the lowest predicted SPF
- Rank correlation: Spearman correlation between predicted and measured SPF
- R2 on test homes
- 95% intervals: 2,000 bootstrap resamples of the 428 homes. Paired comparisons use the same resamples for both methods.

Missing inputs are filled with training-fold medians. The automatic choice takes the model with the lowest inner cross-validation error.

In [ ]:
def precision_flagged(pred, weak_part, share=0.25):
    n_flag = int(round(share * len(pred)))
    flagged = np.argsort(pred, kind='stable')[:n_flag]
    return 100 * weak_part[flagged].mean()


def q1_scores(pred_matrix, idx=None):
    # mean over repeats of (precision of flagged homes, rank correlation)
    if idx is None:
        idx = np.arange(len(spf))
    spf_rank = rankdata(spf[idx])
    precisions = []
    ranks = []
    for r in range(pred_matrix.shape[1]):
        p = pred_matrix[idx, r]
        precisions.append(precision_flagged(p, weak[idx]))
        ranks.append(np.corrcoef(spf_rank, rankdata(p))[0, 1])
    return np.mean(precisions), np.mean(ranks)


def r2_mean(pred_matrix):
    return np.mean([r2_score(spf, pred_matrix[:, r]) for r in range(pred_matrix.shape[1])])


boot_idx = np.random.default_rng(1).integers(0, len(spf), size=(B, len(spf)))


def ci(values):
    return np.percentile(values, 2.5), np.percentile(values, 97.5)


def boot_scores(pred_matrix):
    return np.array([q1_scores(pred_matrix, i) for i in boot_idx])


def run_all(inputs, names=model_names):
    n = len(data)
    test_pred = {}
    for name in names:
        test_pred[name] = np.zeros((n, REPEATS))
    if len(names) > 1:
        test_pred['automatic choice'] = np.zeros((n, REPEATS))
    train_r2 = {name: [] for name in names}
    choices = []
    settings = []
    for r in range(REPEATS):
        for train_idx, test_idx in KFold(n_splits=5, shuffle=True, random_state=r).split(data):
            train = data.iloc[train_idx]
            test = data.iloc[test_idx]
            medians = train[inputs].median()
            X_train = train[inputs].fillna(medians).fillna(0)
            X_test = test[inputs].fillna(medians).fillna(0)
            y_train = train['SPF']
            inner_error = {}
            for name in names:
                model = makers[name]()
                model.fit(X_train, y_train)
                test_pred[name][test_idx, r] = model.predict(X_test)
                train_r2[name].append(r2_score(y_train, model.predict(X_train)))
                if name == 'Ridge':
                    inner = cross_val_score(make_ridge(), X_train, y_train, cv=inner_cv, scoring='neg_mean_squared_error')
                    inner_error[name] = -inner.mean()
                else:
                    inner_error[name] = -model.best_score_
                    settings.append({'model': name, **model.best_params_})
            if len(names) > 1:
                best = min(inner_error, key=inner_error.get)
                choices.append(best)
                test_pred['automatic choice'][test_idx, r] = test_pred[best][test_idx, r]
        print('   repeat', r + 1, 'of', REPEATS, 'done')
    return {'pred': test_pred, 'train_r2': train_r2, 'choices': choices, 'settings': settings}


def results_table(run):
    rows = []
    boot = {}
    n_flag = int(round(0.25 * len(spf)))
    for name, pred in run['pred'].items():
        point = q1_scores(pred)
        boot[name] = boot_scores(pred)
        p_low, p_high = ci(boot[name][:, 0])
        r_low, r_high = ci(boot[name][:, 1])
        if name in run['train_r2']:
            r2_train = np.mean(run['train_r2'][name])
        else:
            r2_train = np.nan
        rows.append({'model': name, 'R2 train': r2_train, 'R2 test': r2_mean(pred),
                     'rank corr': point[1], 'rank low': r_low, 'rank high': r_high,
                     'precision (%)': point[0], 'low': p_low, 'high': p_high,
                     'times random': point[0] / base_rate,
                     'weak found (%)': point[0] * n_flag / weak.sum()})
    return pd.DataFrame(rows).set_index('model'), boot


def paired(boot_a, boot_b, pred_a, pred_b, label):
    diff = boot_a - boot_b
    point = np.array(q1_scores(pred_a)) - np.array(q1_scores(pred_b))
    p_low, p_high = ci(diff[:, 0])
    r_low, r_high = ci(diff[:, 1])
    return {'comparison': label,
            'precision change (points)': point[0], 'low': p_low, 'high': p_high,
            'better (% of resamples)': 100 * (diff[:, 0] > 0).mean(),
            'rank corr change': point[1], 'rank low': r_low, 'rank high': r_high,
            'rank better (% of resamples)': 100 * (diff[:, 1] > 0).mean()}


print('tools ready')

## Step 5: Run everything (about 20 minutes)

All three models on all three input sets, starting with `both`. A progress line is printed after each repeat.

A manufacturer-only baseline is added (Ridge on `brand_Daikin` and `brand_Mitsubishi`). The weak share differs a lot between manufacturers, so this simple rule already beats random flagging, and any model should be compared with it.

In [ ]:
runs = {}
tables = {}
boots = {}
for set_name in ['both', 'design', 'electricity']:
    print('input set:', set_name)
    runs[set_name] = run_all(input_sets[set_name])
    tables[set_name], boots[set_name] = results_table(runs[set_name])

print('input set: manufacturer only (Ridge)')
runs['manufacturer'] = run_all(['brand_Daikin', 'brand_Mitsubishi'], names=['Ridge'])
tables['manufacturer'], boots['manufacturer'] = results_table(runs['manufacturer'])
print('all done')

## Step 6: Model comparison (all inputs)

- Table 1: R2 on training and test homes, rank correlation and flagged precision with 95% intervals. A large gap between training and test R2 means overfitting.
- Table 2: each model minus Ridge on the same homes. An interval that includes zero means no reliable difference.
- Table 3: how often the automatic choice picked each model across 50 splits.

The last line applies the pre-specified rule and names the main model.

In [ ]:
table_both = tables['both']
display(table_both.round(3))

rows = []
for name in ['Random forest', 'LightGBM', 'automatic choice']:
    rows.append(paired(boots['both'][name], boots['both']['Ridge'], runs['both']['pred'][name],
                       runs['both']['pred']['Ridge'], name + ' minus Ridge'))
model_pairs = pd.DataFrame(rows).set_index('comparison')
display(model_pairs.round(3))

counts = pd.Series(runs['both']['choices']).value_counts().reindex(model_names).fillna(0).astype(int)
auto_choice = pd.DataFrame({'times chosen': counts, 'share (%)': 100 * counts / counts.sum()})
display(auto_choice)

better = []
for name in ['Random forest', 'LightGBM']:
    row = model_pairs.loc[name + ' minus Ridge']
    if row['low'] > 0 or row['rank low'] > 0:
        better.append(name)
if better:
    MAIN_MODEL = max(better, key=lambda n: table_both.loc[n, 'rank corr'])
else:
    MAIN_MODEL = 'Ridge'
print('models clearly better than Ridge:', better if better else 'none')
print('MAIN MODEL by the rule written in advance:', MAIN_MODEL)

## Step 7: Input sets (main model)

- Table 1: the manufacturer-only baseline and the main model with each input set
- Table 2: paired differences (all minus manufacturer only, all minus design only, all minus electricity only)
- Table 3: every model on every input set, for the appendix

The two metrics can disagree: flagged precision only looks at the top of the list, while rank correlation looks at the whole ranking.

In [ ]:
rows = []
baseline = tables['manufacturer'].loc['Ridge'].copy()
baseline.name = 'manufacturer only'
rows.append(baseline)
for set_name in ['design', 'electricity', 'both']:
    row = tables[set_name].loc[MAIN_MODEL].copy()
    row.name = set_name
    rows.append(row)
set_table = pd.DataFrame(rows)
display(set_table.round(3))

set_pairs = pd.DataFrame([
    paired(boots['both'][MAIN_MODEL], boots['manufacturer']['Ridge'], runs['both']['pred'][MAIN_MODEL],
           runs['manufacturer']['pred']['Ridge'], 'both minus manufacturer only'),
    paired(boots['both'][MAIN_MODEL], boots['design'][MAIN_MODEL], runs['both']['pred'][MAIN_MODEL],
           runs['design']['pred'][MAIN_MODEL], 'both minus design'),
    paired(boots['both'][MAIN_MODEL], boots['electricity'][MAIN_MODEL], runs['both']['pred'][MAIN_MODEL],
           runs['electricity']['pred'][MAIN_MODEL], 'both minus electricity')]).set_index('comparison')
display(set_pairs.round(3))

full = []
for set_name in ['manufacturer', 'design', 'electricity', 'both']:
    part = tables[set_name][['R2 train', 'R2 test', 'rank corr', 'precision (%)', 'low', 'high']].copy()
    part.insert(0, 'inputs', set_name)
    full.append(part.reset_index())
full_table = pd.DataFrame(pd.concat(full)).set_index(['inputs', 'model'])
full_table.round(3)

## Step 8: Selected tuning settings

How often each setting was chosen for the random forest and LightGBM across 50 splits. If the simplest settings win most of the time, extra complexity is not helping.

In [ ]:
chosen = pd.DataFrame(runs['both']['settings'])
forest_chosen = chosen[chosen['model'] == 'Random forest'].astype({'min_samples_leaf': int})
lightgbm_chosen = chosen[chosen['model'] == 'LightGBM'].astype({'num_leaves': int, 'min_child_samples': int})
display(forest_chosen.groupby(['min_samples_leaf', 'max_features']).size().to_frame('times chosen'))
lightgbm_chosen.groupby(['num_leaves', 'min_child_samples']).size().to_frame('times chosen')

## Step 9: Main figure

(a) Flagged precision with 95% intervals for the manufacturer-only baseline (grey) and the three input sets (blue). Dashed line: random flagging.

(b) Predicted SPF (mean of 10 repeats, always on unseen homes) against measured SPF, using all inputs. Red line: weak threshold. Vertical line: flagging cut-off.

In [ ]:
labels = {'manufacturer only': 'Manufacturer only (baseline)', 'design': 'Design data only',
          'electricity': 'Electricity only', 'both': 'Design + electricity'}
mean_pred = runs['both']['pred'][MAIN_MODEL].mean(axis=1)
cut = np.sort(mean_pred)[int(round(0.25 * len(mean_pred))) - 1]

fig = plt.figure(figsize=(11, 4.2))
plt.subplot(1, 2, 1)
part = set_table.loc[['manufacturer only', 'electricity', 'design', 'both']]
err = [part['precision (%)'] - part['low'], part['high'] - part['precision (%)']]
plt.barh([labels[s] for s in part.index], part['precision (%)'], xerr=err, capsize=3, color=[GREY, BLUE, BLUE, BLUE])
plt.axvline(base_rate, color='black', linestyle='--', linewidth=1, label=f'Random flagging ({base_rate:.1f}%)')
plt.gca().invert_yaxis()
plt.xlim(0, 90)
plt.xlabel('Flagged homes with a weak heat pump (%)')
plt.title(f'(a) Flagging the lowest 25% ({MAIN_MODEL})')
plt.legend(loc='lower right')

plt.subplot(1, 2, 2)
plt.scatter(mean_pred[~weak], spf[~weak], s=9, color=GREY, alpha=0.7, label='Not weak')
plt.scatter(mean_pred[weak], spf[weak], s=9, color=ORANGE, alpha=0.9, label='Weak (SPF < 2.5)')
plt.axhline(2.5, color='red', linestyle=':', linewidth=1)
plt.axvline(cut, color='black', linestyle='--', linewidth=1, label='Flagging cut-off')
plt.xlabel('Predicted SPF (unseen homes)')
plt.ylabel('Measured SPF')
plt.title(f"(b) Predicted vs measured, R2 = {set_table.loc['both', 'R2 test']:.2f}")
plt.legend(loc='upper left')
plt.tight_layout()
plt.show()

# ---- save + download this figure (comment out the 2 lines below after the first run) ----
fig.savefig(FIG_DIR + '/q1_03_step09_main_result.png', dpi=300, bbox_inches='tight')
files.download(FIG_DIR + '/q1_03_step09_main_result.png')

## Step 10: Model comparison figure

(a) R2 on training (grey) and test (blue) homes.

(b) Flagged precision with 95% intervals for each model and the automatic choice.

In [ ]:
y3 = np.arange(3)
four = ['Ridge', 'Random forest', 'LightGBM', 'automatic choice']

fig = plt.figure(figsize=(11, 3.8))
plt.subplot(1, 2, 1)
plt.barh(y3 - 0.2, table_both.loc[model_names, 'R2 train'], height=0.38, color=GREY, label='Training homes')
plt.barh(y3 + 0.2, table_both.loc[model_names, 'R2 test'], height=0.38, color=BLUE, label='Unseen homes')
plt.yticks(y3, model_names)
plt.gca().invert_yaxis()
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Variance in SPF explained (R2)')
plt.title('(a) Training vs unseen homes')
plt.legend(loc='lower right')

plt.subplot(1, 2, 2)
part = table_both.loc[four]
err = [part['precision (%)'] - part['low'], part['high'] - part['precision (%)']]
plt.barh(['Ridge', 'Random forest', 'LightGBM', 'Automatic choice'], part['precision (%)'], xerr=err, capsize=3, color=BLUE)
plt.axvline(base_rate, color='black', linestyle='--', linewidth=1, label=f'Random flagging ({base_rate:.1f}%)')
plt.gca().invert_yaxis()
plt.xlim(0, 90)
plt.xlabel('Flagged homes with a weak heat pump (%)')
plt.title('(b) Flagging the lowest 25%, design + electricity')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

# ---- save + download this figure (comment out the 2 lines below after the first run) ----
fig.savefig(FIG_DIR + '/q1_03_step10_model_comparison.png', dpi=300, bbox_inches='tight')
files.download(FIG_DIR + '/q1_03_step10_model_comparison.png')

## Step 11: Learning curve (main model, all inputs)

Would more homes help? The model is trained on subsets of the training homes and tested on the same test homes. A test curve that is still rising means more data would help. A flat curve with a small train-test gap means the limit is the information in the inputs.

In [ ]:
inputs = input_sets['both']
smallest_train = len(data) - int(np.ceil(len(data) / 5))
sizes = [n for n in [40, 80, 160, 240] if n < smallest_train] + [smallest_train]

rows = []
for n in sizes:
    for r in range(REPEATS):
        pred = np.zeros(len(data))
        train_scores = []
        for fold, (train_idx, test_idx) in enumerate(KFold(n_splits=5, shuffle=True, random_state=r).split(data)):
            chosen_idx = np.random.default_rng(1000 * r + fold).choice(train_idx, size=n, replace=False)
            train = data.iloc[chosen_idx]
            test = data.iloc[test_idx]
            medians = train[inputs].median()
            model = makers[MAIN_MODEL]()
            model.fit(train[inputs].fillna(medians).fillna(0), train['SPF'])
            pred[test_idx] = model.predict(test[inputs].fillna(medians).fillna(0))
            train_pred = model.predict(train[inputs].fillna(medians).fillna(0))
            train_scores.append([r2_score(train['SPF'], train_pred),
                                 precision_flagged(train_pred, train['SPF'].values < 2.5)])
        train_scores = np.mean(train_scores, axis=0)
        rows.append({'training homes': n, 'repeat': r, 'R2 train': train_scores[0], 'R2 test': r2_score(spf, pred),
                     'precision train (%)': train_scores[1], 'precision test (%)': precision_flagged(pred, weak)})
    print(n, 'training homes: done')

curve = pd.DataFrame(rows)
curve_mean = curve.groupby('training homes').mean().drop(columns='repeat')
curve_sd = curve.groupby('training homes').std().drop(columns='repeat')
display(curve_mean.round(3))

fig = plt.figure(figsize=(10, 3.8))
plt.subplot(1, 2, 1)
plt.errorbar(sizes, curve_mean['R2 train'], yerr=curve_sd['R2 train'], marker='o', color=GREY, linestyle='--', capsize=3, label='Training homes')
plt.errorbar(sizes, curve_mean['R2 test'], yerr=curve_sd['R2 test'], marker='o', color=BLUE, capsize=3, label='Unseen homes')
plt.axhline(0, color='black', linewidth=0.8)
plt.xlabel('Number of training homes')
plt.ylabel('Variance in SPF explained (R2)')
plt.title('(a) Explained variance')
plt.legend()
plt.subplot(1, 2, 2)
plt.errorbar(sizes, curve_mean['precision train (%)'], yerr=curve_sd['precision train (%)'], marker='o', color=GREY, linestyle='--', capsize=3, label='Training homes')
plt.errorbar(sizes, curve_mean['precision test (%)'], yerr=curve_sd['precision test (%)'], marker='o', color=BLUE, capsize=3, label='Unseen homes')
plt.axhline(base_rate, color='black', linestyle=':', linewidth=1, label=f'Random flagging ({base_rate:.1f}%)')
plt.xlabel('Number of training homes')
plt.ylabel('Flagged homes that are weak (%)')
plt.title('(b) Flagging the lowest 25%')
plt.legend()
plt.tight_layout()
plt.show()

# ---- save + download this figure (comment out the 2 lines below after the first run) ----
fig.savefig(FIG_DIR + '/q1_03_step11_learning_curve.png', dpi=300, bbox_inches='tight')
files.download(FIG_DIR + '/q1_03_step11_learning_curve.png')

## Step 12: Save

Tables go to the tables folder. Predictions of the main model (every home, every repeat) are saved for notebook 4 as `q1_predictions.csv`.

In [ ]:
table_both.to_csv(TABLE_DIR + '/q1_03_model_comparison.csv')
model_pairs.to_csv(TABLE_DIR + '/q1_03_model_pairs.csv')
auto_choice.to_csv(TABLE_DIR + '/q1_03_automatic_choice.csv')
chosen.to_csv(TABLE_DIR + '/q1_03_tuned_settings.csv', index=False)
set_table.to_csv(TABLE_DIR + '/q1_03_input_sets.csv')
set_pairs.to_csv(TABLE_DIR + '/q1_03_input_set_pairs.csv')
full_table.to_csv(TABLE_DIR + '/q1_03_all_models_all_inputs.csv')
curve_mean.to_csv(TABLE_DIR + '/q1_03_learning_curve.csv')

pred_out = pd.DataFrame(runs['both']['pred'][MAIN_MODEL], columns=[f'repeat_{r}' for r in range(REPEATS)])
pred_out.insert(0, 'Property_ID', data['Property_ID'].values)
pred_out.insert(1, 'model', MAIN_MODEL)
pred_out.to_csv(OUT_DIR + '/q1_predictions.csv', index=False)
with open(OUT_DIR + '/q1_main_model.json', 'w') as f:
    json.dump({'main_model': MAIN_MODEL}, f)
print('saved 8 tables, q1_predictions.csv and q1_main_model.json')